# V5C 替代标的相关性深度测试

> 2026-05-08 · 沉淀期内的认知验证

## 测试目的

针对 VOO 替代候选 + V5C 已用进攻类资产做一次完整的相关性矩阵测试：
- **VOO** (V5C 3.3a 进攻核心)
- **QQQ** (V5C 3.3a 已用 NDX 暴露)
- **VXUS** (V5C 1.0 移除候选)
- **SCHD** (V5C 1.0 移除候选)
- **AVUV** (V5C 1.0 移除候选)

## 三层视角

1. **静态相关性矩阵** — 三个时间窗口（7Y/16Y/23.8Y）下的两两相关性
2. **滚动相关性** — 1Y 滚动窗口看相关性随时间变化
3. **跨类相关性** — 这 5 个股票类资产 vs V5C 3.3a 的对冲资产（GLDM/BCX/DBMF/VGSH）

## 关键问题

1. 这 5 个股票类资产之间是否存在「真分散」对（相关性 < 0.7）？
2. 相关性是否随时间发生结构性变化（如 AVUV 7Y 0.795 vs 历史 0.85）？
3. 哪个标的与 V5C 对冲层资产相关性最低（潜在加分项）？

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

tickers = ['VOO','QQQ','VXUS','SCHD','AVUV',
           'VFINX','EFA','DVY','VISVX',
           'GLD','GC=F','DBC','PCRIX','DBMF','AQRIX','VFITX']
raw = yf.download(tickers, start='1999-01-01', auto_adjust=True, progress=False)['Close']

def synthesize(short, long):
    short = short.dropna(); long = long.dropna()
    if len(short) == 0: return long
    overlap = short.index[0]
    if overlap <= long.index[0]: return short
    long_at = long.loc[:overlap].iloc[-1] if not long.loc[:overlap].empty else long.iloc[0]
    scale = short.iloc[0] / long_at
    early = long.loc[:overlap].iloc[:-1] * scale
    return pd.concat([early, short]).sort_index().pipe(lambda s: s[~s.index.duplicated(keep='last')])

data = pd.DataFrame({
    'VOO': raw['VFINX'],
    'QQQ': raw['QQQ'],
    'VXUS': synthesize(raw['VXUS'], raw['EFA']),
    'SCHD': synthesize(raw['SCHD'], raw['DVY']),
    'AVUV': synthesize(raw['AVUV'], raw['VISVX']),
    'GLDM': synthesize(raw['GLD'], raw['GC=F']),
    'BCX': synthesize(raw['DBC'], raw['PCRIX']),
    'DBMF': raw['DBMF'],
    'AQRIX': raw['AQRIX'],
    'VGSH': raw['VFITX'],
})

for c in data.columns:
    fv = data[c].first_valid_index()
    print(f'{c:8} 起始: {fv.date() if fv else "N/A"}')

## 一、5 个股票类资产的相关性矩阵

三个时间窗口比较

In [ ]:
# 准备日收益率
ret = data.pct_change().dropna(how='all')

stocks = ['VOO', 'QQQ', 'VXUS', 'SCHD', 'AVUV']

# 三个窗口
windows = {
    '7Y (2019-2026)': ret.loc['2019-05-01':, stocks].dropna(),
    '16Y (2010-2026)': ret.loc['2010-09-30':, stocks].dropna(),
    '23.8Y (2002-2026)': ret.loc['2003-11-07':, stocks].dropna(),
}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (name, w) in zip(axes, windows.items()):
    corr = w.corr()
    sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlBu_r', vmin=0.6, vmax=1.0, 
                ax=ax, cbar=True, square=True, annot_kws={'size': 11})
    ax.set_title(f'股票类资产相关性 — {name}', fontsize=12)
plt.tight_layout()
plt.show()

print('\n=== 三窗口相关性数值（vs VOO）===')
print(f"{'对':>15} {'7Y':>10} {'16Y':>10} {'23.8Y':>10}")
for t in ['QQQ', 'VXUS', 'SCHD', 'AVUV']:
    row = f'{t} vs VOO'
    line = f'{row:>15}'
    for w in windows.values():
        line += f' {w[t].corr(w["VOO"]):>10.3f}'
    print(line)

In [ ]:
# 所有两两相关性 (23.8Y)
print('=== 23.8Y 长史相关性矩阵（最严格基准）===\n')
corr_long = windows['23.8Y (2002-2026)'].corr()
print(corr_long.round(3).to_string())

print('\n=== 关键发现：寻找「真分散对」（相关性 < 0.70）===\n')
found = False
for i, t1 in enumerate(stocks):
    for j, t2 in enumerate(stocks):
        if j > i:
            for label, w in windows.items():
                c = w[t1].corr(w[t2])
                if c < 0.70:
                    print(f'  {t1} vs {t2}：{c:.3f} （{label}）⭐ 接近真分散')
                    found = True
if not found:
    print('  ⚠️ 没有任何两个股票类资产相关性 < 0.70')
    print('  即：这 5 个标的之间「都是同一类资产」，无法在内部实现真分散')

## 二、滚动 1Y 相关性 — 看时变特征

In [ ]:
# 滚动 1Y (252 交易日) 相关性 vs VOO
ret_long = ret[stocks].dropna()
print(f'用于滚动相关性的数据：{ret_long.index[0].date()} → {ret_long.index[-1].date()}')

fig, ax = plt.subplots(figsize=(15, 6))
for t in ['QQQ', 'VXUS', 'SCHD', 'AVUV']:
    rolling_corr = ret_long[t].rolling(252).corr(ret_long['VOO'])
    ax.plot(rolling_corr, label=f'{t} vs VOO', linewidth=1.5, alpha=0.8)

ax.axhline(0.70, color='red', linestyle='--', alpha=0.5, label='真分散阈值 (0.70)')
ax.axhline(0.85, color='orange', linestyle='--', alpha=0.5, label='高相关阈值 (0.85)')
ax.set_title('滚动 1Y 相关性 vs VOO （时变特征）', fontsize=13)
ax.set_ylabel('相关系数')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
ax.set_ylim(0.4, 1.0)
plt.tight_layout()
plt.show()

In [ ]:
# 滚动相关性的最低/最高/最近
print('=== 滚动 1Y 相关性 vs VOO 统计 ===\n')
print(f"{'标的':<8} {'最低':>10} {'最低发生':>14} {'最高':>10} {'最高发生':>14} {'最近 1Y':>12} {'变化趋势':>16}")
print('-' * 92)
for t in ['QQQ', 'VXUS', 'SCHD', 'AVUV']:
    rolling_corr = ret_long[t].rolling(252).corr(ret_long['VOO']).dropna()
    min_v = rolling_corr.min()
    min_d = rolling_corr.idxmin().date()
    max_v = rolling_corr.max()
    max_d = rolling_corr.idxmax().date()
    recent = rolling_corr.iloc[-1]
    # 比较最近 1Y vs 历史中位数
    median = rolling_corr.median()
    trend = '↑ 上升' if recent > median + 0.05 else ('↓ 下降' if recent < median - 0.05 else '→ 持平')
    print(f'{t:<8} {min_v:>10.3f} {str(min_d):>14} {max_v:>10.3f} {str(max_d):>14} {recent:>12.3f} {trend:>16}')

## 三、与 V5C 对冲层的相关性（真正的「分散维度」）

对比这 5 个股票类资产 vs V5C 3.3a 4 个对冲/防御资产（GLDM/BCX/DBMF/VGSH）

In [ ]:
# 5 stocks × 4 hedges 矩阵
hedges = ['GLDM', 'BCX', 'DBMF', 'VGSH']

# 7Y 数据 (DBMF 实际)
ret_7Y_full = ret.loc['2019-05-01':, stocks + hedges].dropna()
print(f'7Y 数据期: {ret_7Y_full.index[0].date()} → {ret_7Y_full.index[-1].date()}')
print()

cross_corr = pd.DataFrame(index=stocks, columns=hedges, dtype=float)
for s in stocks:
    for h in hedges:
        cross_corr.loc[s, h] = ret_7Y_full[s].corr(ret_7Y_full[h])

print('=== 股票类 vs 对冲层相关性 (7Y) ===\n')
print(cross_corr.round(3).to_string())

print('\n=== 解读：哪个股票标的最「适合」加进 V5C? ===')
print('（标准：与对冲资产相关性最低，能与现有对冲层最少干扰）\n')

# 计算每个 stock 与 hedges 的平均绝对相关性
for s in stocks:
    avg_abs_corr = cross_corr.loc[s].abs().mean()
    print(f'  {s}: 平均与对冲层相关性 = {avg_abs_corr:.3f}')

In [ ]:
# 可视化
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cross_corr.astype(float), annot=True, fmt='.3f', 
            cmap='RdBu_r', vmin=-0.3, vmax=0.5, ax=ax,
            cbar_kws={'label': '相关系数'},
            annot_kws={'size': 12})
ax.set_title('股票类资产 vs V5C 对冲层相关性 (7Y)', fontsize=13)
ax.set_ylabel('股票类资产')
ax.set_xlabel('V5C 对冲/防御层')
plt.tight_layout()
plt.show()

## 四、危机时期相关性「同步上升」效应

金融危机时所有股票相关性会同步上升到 0.95+。这是 V5C 设计的关键约束。

In [ ]:
# 关键危机时期内的相关性
crises = {
    '2008 GFC':       ('2008-09-01', '2009-03-31'),
    '2020 COVID':     ('2020-02-19', '2020-04-30'),
    '2022 Bear':      ('2022-01-01', '2022-12-31'),
}

# 平静期对照（2017-2019）
calm = ('2017-01-01', '2019-12-31')

print('=== 危机期间股票类相关性矩阵 vs VOO ===\n')
print(f"{'时期':<20} {'QQQ':>8} {'VXUS':>8} {'SCHD':>8} {'AVUV':>8}")
print('-' * 60)

# 平静期
calm_ret = ret.loc[calm[0]:calm[1], stocks].dropna()
line = f'{"平静期 (17-19)":<20}'
for t in ['QQQ', 'VXUS', 'SCHD', 'AVUV']:
    if t in calm_ret.columns:
        c = calm_ret[t].corr(calm_ret['VOO'])
        line += f' {c:>8.3f}'
    else:
        line += f' {"N/A":>8}'
print(line)

# 各危机
for cn, (s, e) in crises.items():
    crisis_ret = ret.loc[s:e, stocks].dropna()
    line = f'{cn:<20}'
    for t in ['QQQ', 'VXUS', 'SCHD', 'AVUV']:
        if t in crisis_ret.columns:
            c = crisis_ret[t].corr(crisis_ret['VOO'])
            line += f' {c:>8.3f}'
        else:
            line += f' {"N/A":>8}'
    print(line)

In [ ]:
# 危机时对冲层 vs 股票相关性也同步上升吗？
print('=== 危机期间 V5C 对冲层 vs VOO 相关性 ===\n')
print(f"{'时期':<20} {'GLDM':>8} {'BCX':>8} {'DBMF':>8} {'VGSH':>8}")
print('-' * 60)

calm_ret_full = ret.loc[calm[0]:calm[1], stocks + hedges].dropna()
line = f'{"平静期 (17-19)":<20}'
for t in hedges:
    c = calm_ret_full[t].corr(calm_ret_full['VOO']) if t in calm_ret_full.columns else None
    line += f' {c:>8.3f}' if c is not None else f' {"N/A":>8}'
print(line)

for cn, (s, e) in crises.items():
    crisis_full = ret.loc[s:e, stocks + hedges].dropna()
    line = f'{cn:<20}'
    for t in hedges:
        if t in crisis_full.columns and len(crisis_full) > 5:
            c = crisis_full[t].corr(crisis_full['VOO'])
            line += f' {c:>8.3f}'
        else:
            line += f' {"N/A":>8}'
    print(line)

print('\n💡 解读：危机时股票类相关性 → 1.0（同涨同跌），')
print('   但对冲层（GLDM/DBMF）相关性可能保持低或转负 — 这是真分散的价值')

## 五、综合分析

综合前面所有数据，回答三个核心问题：

### Q1: 5 个股票类资产之间是否有「真分散」对？
（运行后填写）

### Q2: 相关性是否随时间结构性变化？
（看滚动 1Y 图 + 三窗口对比）

### Q3: 哪个标的与 V5C 对冲层相关性最低？
（看 cross_corr 矩阵）

## 六、对 V5C 4.0 设计的输入

基于本次相关性深度测试，V5C 4.0 设计应考虑：

1. **进攻层内部分散是伪命题** — 5 个股票之间相关性都 > 0.7
2. **真正的 Sharpe 增量来自对冲层** — 已经在 V5C 3.3a 中通过 GLDM/BCX/DBMF 实现
3. **如果 AVUV 相关性持续下降** → 未来某个时点可作为「准真分散」候选加入
4. **危机时所有股票相关性 → 1.0** — 永远不要指望股票内部分散在危机时起作用